# Grafo da loja

Monta o grafo físico da loja, calcula as distâncias com Dijkstra e lê os layouts e os pedidos

## Parâmetros

A geometria da loja é premissa do grupo, pois o Instacart não traz a planta: corredores de 20 m com eixos a 4 m, slots a cada 2,5 m sobre o eixo (o primeiro a 1,25 m da frente) e transversais na frente ($y = 0$) e no fundo ($y = 20$ m). Os tempos (1 m/s, 15 s por item, 120 s fixos por pedido e turno de 8 h) só entram no cálculo de tempo e de pedidos por turno, não na escolha da rota. `PEDIDOS_PRIOR` é o total de pedidos do histórico, o $N$ do lift.

In [1]:
import heapq
from pathlib import Path

import numpy as np
import pandas as pd

PASTA_PROJETO = Path.cwd().parent
PASTA_DADOS = PASTA_PROJETO / "dados"
PASTA_RESULTADOS = PASTA_PROJETO / "resultados"
ARQUIVOS_LAYOUT = {
    "L0": PASTA_DADOS / "layouts" / "layout_L0.csv",
    "L1": PASTA_DADOS / "layouts" / "layout_L1_final.csv",
    "L2": PASTA_DADOS / "layouts" / "layout_L2_final.csv",
}
LAYOUTS = list(ARQUIVOS_LAYOUT)
PEDIDOS_PRIOR = 3_214_874

DISTANCIA_ENTRE_CORREDORES = 4.0
DISTANCIA_ENTRE_SLOTS = 2.5
ALTURA_PRIMEIRO_SLOT = 1.25
ALTURA_FRENTE = 0.0
ALTURA_FUNDO = 20.0

VELOCIDADE = 1.0
SEGUNDOS_POR_ITEM = 15.0
SEGUNDOS_POR_PEDIDO = 120.0
DURACAO_TURNO = 8 * 3600

## Grafo físico

- Os primeiros vértices são os slots (corredor, posição), depois ENTRADA e CAIXA.
- Cada slot liga a um nó de acesso no eixo do corredor, e os corredores se ligam pelas transversais da frente e do fundo.
- O peso de cada aresta é a distância em metros.

A distância entre dois pontos é o caminho mínimo nesse grafo, calculado pelo Dijkstra. Como todas as rotas só visitam slots, ENTRADA e CAIXA, o Dijkstra roda uma vez a partir de cada um desses 136 vértices e gera a matriz `distancias` (136 × 136); depois disso, medir uma rota é só somar valores da matriz. `caminho` reconstrói o trajeto pelos corredores, usado apenas para desenhar as rotas.

In [2]:
class Loja:
    def __init__(self, slots):
        self.slots = sorted(slots)
        self.n_slots = len(self.slots)
        self.ENTRADA = self.n_slots
        self.CAIXA = self.n_slots + 1
        self.corredor = np.array([c for c, _ in self.slots])
        self.posicao = np.array([p for _, p in self.slots])
        self.indice = {slot: i for i, slot in enumerate(self.slots)}
        self.coordenadas = []
        self.vizinhos = []
        self._montar_grafo()

    def indice_slot(self, corredor, posicao):
        return self.indice[(corredor, posicao)]

    def _novo_vertice(self, x, y):
        self.coordenadas.append((x, y))
        self.vizinhos.append([])
        return len(self.vizinhos) - 1

    def _ligar(self, a, b):
        (xa, ya), (xb, yb) = self.coordenadas[a], self.coordenadas[b]
        distancia = abs(xa - xb) + abs(ya - yb)
        self.vizinhos[a].append((b, distancia))
        self.vizinhos[b].append((a, distancia))

    def _montar_grafo(self):
        corredores = sorted(set(self.corredor))
        x = lambda corredor: (corredor - 1) * DISTANCIA_ENTRE_CORREDORES
        y = lambda posicao: ALTURA_PRIMEIRO_SLOT + (posicao - 1) * DISTANCIA_ENTRE_SLOTS

        for corredor, posicao in self.slots:
            self._novo_vertice(x(corredor), y(posicao))
        entrada = self._novo_vertice(x(corredores[0]), ALTURA_FRENTE)
        caixa = self._novo_vertice(x(corredores[-1]), ALTURA_FRENTE)

        frente, fundo = {}, {}
        for corredor in corredores:
            frente[corredor] = self._novo_vertice(x(corredor), ALTURA_FRENTE)
            fundo[corredor] = self._novo_vertice(x(corredor), ALTURA_FUNDO)
            caminho = [frente[corredor]]
            for posicao in sorted(p for c, p in self.slots if c == corredor):
                acesso = self._novo_vertice(x(corredor), y(posicao))
                self._ligar(self.indice[(corredor, posicao)], acesso)
                caminho.append(acesso)
            caminho.append(fundo[corredor])
            for a, b in zip(caminho, caminho[1:]):
                self._ligar(a, b)

        for atual, proximo in zip(corredores, corredores[1:]):
            self._ligar(frente[atual], frente[proximo])
            self._ligar(fundo[atual], fundo[proximo])
        self._ligar(entrada, frente[corredores[0]])
        self._ligar(caixa, frente[corredores[-1]])

    def dijkstra(self, origem):
        distancia = [float("inf")] * len(self.vizinhos)
        anterior = [None] * len(self.vizinhos)
        distancia[origem] = 0.0
        fila = [(0.0, origem)]
        while fila:
            d, atual = heapq.heappop(fila)
            if d > distancia[atual]:
                continue
            for vizinho, peso in self.vizinhos[atual]:
                if d + peso < distancia[vizinho]:
                    distancia[vizinho] = d + peso
                    anterior[vizinho] = atual
                    heapq.heappush(fila, (d + peso, vizinho))
        return distancia, anterior

    def matriz_distancias(self):
        n = self.n_slots + 2
        return np.array([self.dijkstra(i)[0][:n] for i in range(n)])

    def caminho(self, origem, destino):
        _, anterior = self.dijkstra(origem)
        vertices = [destino]
        while vertices[-1] != origem:
            vertices.append(anterior[vertices[-1]])
        return [self.coordenadas[v] for v in reversed(vertices)]

## Layouts, pedidos e camada lógica G2

Os três layouts usam os mesmos 134 slots, então a loja é montada uma vez só. As seções de cada pedido ficam na ordem em que entraram no carrinho.

A camada lógica G2 liga duas seções quando elas aparecem juntas em pelo menos `n_min` pedidos do prior, com peso igual ao lift. No  arquivo de coocorrência observa-se que todos os pedidos inclui o train, onde estão os pedidos de avaliação, por isso a contagem do train é descontada.

In [3]:
def carregar_layout(nome):
    tabela = pd.read_csv(ARQUIVOS_LAYOUT[nome])
    return {int(linha.aisle_id): (int(linha.corredor), int(linha.posicao_na_frente))
            for linha in tabela.itertuples()}


def nomes_das_secoes():
    tabela = pd.read_csv(PASTA_DADOS / "curados" / "catalogo_aisles_departamentos.csv")
    return dict(zip(tabela["aisle_id"], tabela["aisle"]))


def carregar_pedidos():
    pedidos = pd.read_csv(PASTA_DADOS / "amostras" / "Pedidos.csv", encoding="utf-8-sig")
    pedidos = pedidos.rename(columns={"itens no pedido": "n_itens"})[["id_teste", "order_id", "n_itens"]]

    itens = pd.read_csv(PASTA_DADOS / "amostras" / "pedidos_avaliacao_carrinho.csv")
    itens = itens.sort_values(["order_id", "add_to_cart_order"]).drop_duplicates(["order_id", "aisle_id"])
    pedidos["secoes"] = pedidos["order_id"].map(itens.groupby("order_id")["aisle_id"].apply(list))
    return pedidos


def arestas_g2(n_min=300):
    chave = ["aisle_id_1", "aisle_id_2"]
    todos = pd.read_csv(PASTA_DADOS / "curados" / "coocorrencia_aisles_prior_train.csv")
    so_train = pd.read_csv(PASTA_DADOS / "curados" / "coocorrencia_aisles_train.csv")
    pares = todos.merge(so_train[chave + ["quantidade_pedidos_juntos"]], on=chave, how="left",
                        suffixes=("", "_train"))
    pares["n_ij"] = pares["quantidade_pedidos_juntos"] - pares["quantidade_pedidos_juntos_train"].fillna(0)

    demanda = pd.read_csv(PASTA_DADOS / "curados" / "frequencia_aisles.csv").set_index("aisle_id")["pedidos_prior"]
    produto = pares["aisle_id_1"].map(demanda) * pares["aisle_id_2"].map(demanda)
    pares["lift"] = pares["n_ij"] * PEDIDOS_PRIOR / produto
    return pares.loc[pares["n_ij"] >= n_min, chave + ["n_ij", "lift"]].reset_index(drop=True)


def slots_do_pedido(secoes, layout, loja):
    return list(dict.fromkeys(loja.indice_slot(*layout[secao]) for secao in secoes))


def tempo_coleta(distancia, n_itens):
    return distancia / VELOCIDADE + SEGUNDOS_POR_ITEM * n_itens + SEGUNDOS_POR_PEDIDO


loja = Loja(list(carregar_layout("L0").values()))
distancias = loja.matriz_distancias()
pedidos = carregar_pedidos()